# SAM Downstream Dependencies Parser
This notebook:
1. Connects to rhnv-edwprod.rhnv.hosted with Windows Authentication
2. Queries SummaryBindingDependency for ClaimDOSMartHomeTownHealth
3. Parses SQL from AttributeValueLongTXT to extract tables and columns
4. Writes parsed results to rhnv-edwdev.rhnv.hosted SAM.ClientAdmin.DownstreamTablesColumnsBASE

In [72]:
import pyodbc
import pandas as pd
import re
from sqlparse import parse, sql
from sqlparse.tokens import Keyword, Name
import sqlparse

In [73]:
# Connection strings
prod_server = 'rhnv-edwprod.rhnv.hosted'
dev_server = 'rhnv-edwdev.rhnv.hosted'

# Production connection (source)
prod_conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={prod_server};Trusted_Connection=yes;'
prod_conn = pyodbc.connect(prod_conn_str)

# Development connection (destination)
dev_conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={dev_server};DATABASE=SAM;Trusted_Connection=yes;'
dev_conn = pyodbc.connect(dev_conn_str)

print("Connected to both servers successfully")

Connected to both servers successfully


In [74]:
# Query to get binding dependencies
query = """
SELECT 
    sbd.SourceDataMartNM,
    sbd.BindingClassificationCD,
    sbd.SourceEntityNM,
    sbd.DestinationDataMartNM,
    sbd.DestinationBindingID,
    oab.AttributeValueLongTXT,
    sbd.DestinationBindingNM,
    sbd.DestinationEntityNM
FROM EDWAdmin.CatalystAdmin.SummaryBindingDependency sbd
JOIN EDWAdmin.CatalystAdmin.ObjectAttributeBASE oab 
    ON sbd.DestinationBindingID = oab.ObjectID 
    AND oab.AttributeNM = 'UserDefinedSQL'
WHERE 1=1
    AND sbd.SourceDataMartNM = 'ClaimDOSMartHomeTownHealth'
    AND sbd.SourceDataMartNM <> sbd.DestinationDataMartNM
ORDER BY sbd.DestinationDataMartNM, sbd.DestinationBindingNM
"""

df_bindings = pd.read_sql(query, prod_conn)
print(f"Retrieved {len(df_bindings)} binding records")
df_bindings.head()

C:\Users\58811.WMCDOMAIN\AppData\Local\Temp\ipykernel_34508\3681315393.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_bindings = pd.read_sql(query, prod_conn)


Retrieved 16 binding records


,SourceDataMartNM,BindingClassificationCD,SourceEntityNM,DestinationDataMartNM,DestinationBindingID,AttributeValueLongTXT,DestinationBindingNM,DestinationEntityNM
0,ClaimDOSMartHomeTownHealth,Event,HTHClaimDiagnosis,ClaimDOSMart,2828,\r\n\r\nWITH DiagnosisCTE \r\n\tAS (\r\n\tSELE...,ClaimDiagnosis (Config),ClaimDiagnosis
1,ClaimDOSMartHomeTownHealth,Event,HTHClaimLineItem,ClaimDOSMart,2763,"SELECT\r\n\ta.PayerHierarchyCD\r\n\t,a.PayerCD...",ClaimLineItemStage (Config),ClaimLineItemStage
2,ClaimDOSMartHomeTownHealth,Event,HTHClaimLineItem,ClaimDOSMart,2764,"SELECT\r\n PayerHierarchyCD\r\n ,ClaimID...",ClaimLineItemStageExtension,ClaimLineItemStageExtension
3,ClaimDOSMartHomeTownHealth,ReportingView,HTHContract,ClaimDOSMart,2805,"SELECT\r\n\tPayerHierarchyCD\r\n\t,ContractID\...",ContractView,Contract
4,ClaimDOSMartHomeTownHealth,Event,HTHExtendedProviderGroup,ClaimDOSMart,3346,"SELECT \r\n\tNPI\r\n\t,SpecialtyDSC\r\n\t,Grou...",ExtendedProviderGroup,ExtendedProviderGroup


In [75]:
def parse_sql_for_tables_columns(sql_text):
    """
    Parse SQL to extract database, schema, table, column, and alias information.
    Returns list of dictionaries with parsed components.
    """
    if not sql_text or pd.isna(sql_text):
        return []
    
    results = []
    
    # Pattern to match: [Database].[Schema].[Table] or Schema.Table or just Table
    # Also captures columns with aliases
    table_pattern = r'(?:FROM|JOIN)\s+(?:\[?([\w]+)\]?\.)?(?:\[?([\w]+)\]?\.)?\[?([\w]+)\]?'
    column_pattern = r'(?:SELECT|,)\s+(?:\[?([\w]+)\]?\.)?(?:\[?([\w]+)\]?\.)?\[?([\w]+)\]?(?:\s+(?:AS\s+)?\[?([\w]+)\]?)?'
    
    # Find all table references
    tables = re.finditer(table_pattern, sql_text, re.IGNORECASE)
    table_info = {}
    
    for match in tables:
        db = match.group(1)
        schema = match.group(2) if match.group(2) else match.group(1)
        table = match.group(3) if match.group(2) else match.group(2)
        
        if not table:
            table = match.group(3)
            schema = match.group(2)
            db = match.group(1)
        
        table_key = table.lower() if table else None
        if table_key:
            table_info[table_key] = {'database': db, 'schema': schema, 'table': table}
    
    # Find all column references
    columns = re.finditer(column_pattern, sql_text, re.IGNORECASE)
    
    for match in columns:
        table_alias = match.group(1)
        potential_schema = match.group(2)
        column = match.group(3)
        alias = match.group(4)
        
        # Try to resolve table information
        db_name = None
        schema_name = None
        table_name = None
        
        if table_alias and table_alias.lower() in table_info:
            db_name = table_info[table_alias.lower()]['database']
            schema_name = table_info[table_alias.lower()]['schema']
            table_name = table_info[table_alias.lower()]['table']
        elif potential_schema:
            schema_name = table_alias
            table_name = potential_schema
        
        results.append({
            'ParsedDatabaseNM': db_name,
            'ParsedSchemaNM': schema_name,
            'ParsedTableNM': table_name,
            'ParsedColumnNM': column,
            'ParsedColumnAliasNM': alias
        })
    
    return results

# Test the parser on first record
if len(df_bindings) > 0:
    test_sql = df_bindings.iloc[0]['AttributeValueLongTXT']
    test_results = parse_sql_for_tables_columns(test_sql)
    print(f"Test parse found {len(test_results)} column references")
    if test_results:
        print("Sample:", test_results[0])

Test parse found 6 column references
Sample: {'ParsedDatabaseNM': None, 'ParsedSchemaNM': None, 'ParsedTableNM': None, 'ParsedColumnNM': 'CAST', 'ParsedColumnAliasNM': None}


In [76]:
# Parse all SQL statements
all_parsed_results = []

for idx, row in df_bindings.iterrows():
    binding_id = row['DestinationBindingID']
    sql_text = row['AttributeValueLongTXT']
    
    parsed = parse_sql_for_tables_columns(sql_text)
    
    for item in parsed:
        item['DestinationBindingID'] = binding_id
        all_parsed_results.append(item)

df_parsed = pd.DataFrame(all_parsed_results)
print(f"Parsed {len(df_parsed)} total column references from {len(df_bindings)} bindings")
df_parsed.head(10)

Parsed 66 total column references from 16 bindings


,ParsedDatabaseNM,ParsedSchemaNM,ParsedTableNM,ParsedColumnNM,ParsedColumnAliasNM,DestinationBindingID
0,None,None,None,CAST,NaN,2828
1,None,None,None,PayerHierarchyCD,NaN,2828
2,None,None,None,PayerHierarchyCD,NaN,2828
3,None,None,None,PayerHierarchyCD,NaN,2828
4,None,None,None,ICDDiagnosisCD,NaN,2828
5,None,None,None,REPLACE,NaN,2828
6,None,None,None,PayerHierarchyCD,NaN,2763
7,None,None,None,PayerHierarchyCD,NaN,2763
8,None,None,None,PayerHierarchyCD,NaN,2764
9,None,None,None,PayerHierarchyCD,NaN,2805


In [77]:
# Reorder columns to match destination table structure
df_output = df_parsed[[
    'DestinationBindingID',
    'ParsedDatabaseNM',
    'ParsedSchemaNM',
    'ParsedTableNM',
    'ParsedColumnNM',
    'ParsedColumnAliasNM'
]]

print(f"Prepared {len(df_output)} records for insertion")
df_output.head()

Prepared 66 records for insertion


,DestinationBindingID,ParsedDatabaseNM,ParsedSchemaNM,ParsedTableNM,ParsedColumnNM,ParsedColumnAliasNM
0,2828,None,None,None,CAST,NaN
1,2828,None,None,None,PayerHierarchyCD,NaN
2,2828,None,None,None,PayerHierarchyCD,NaN
3,2828,None,None,None,PayerHierarchyCD,NaN
4,2828,None,None,None,ICDDiagnosisCD,NaN


In [78]:
# Check existing schemas
schema_query = "SELECT name FROM sys.schemas ORDER BY name"
df_schemas = pd.read_sql(schema_query, dev_conn)
print("Available schemas:")
print(df_schemas['name'].tolist())

Available schemas:
['ACS', 'Acute', 'AcuteReference', 'AcuteReporting', 'AcuteTherapy', 'AMI', 'AperekAuditing', 'Appointment', 'BehavioralHealth', 'BindingStatus', 'BloodCharges', 'BloodUtilization', 'CABG', 'CallCenter', 'Cardiovascular', 'CareProcessCostingAndFinance', 'CaseManagement', 'CatalystAdmin', 'Chime', 'Chole', 'Cholectomy', 'Claim', 'ClientAdmin', 'Clinical', 'CMSQuality', 'CMSStatistics', 'Colorectal', 'ComparativeAnalytics', 'Config', 'ContactTracing', 'COPD', 'CopyoDiabetes', 'CopyofDiabetes', 'CopyofHR', 'CopyofLaborManagement', 'CorusReporting', 'CostflexCosts', 'COVID', 'COVID19', 'CPN', 'Cy', 'DataCatalogExplorer', 'DataEntryAcuteLocations', 'DataMartComparison', 'DataMartReview', 'db_accessadmin', 'db_backupoperator', 'db_datareader', 'db_datawriter', 'db_ddladmin', 'db_denydatareader', 'db_denydatawriter', 'db_owner', 'db_securityadmin', 'dbo', 'Definition', 'DevClaimsMQS', 'Diabetes', 'Dimension', 'EDWCompare', 'EmergencyDepartment', 'Executive', 'Finance', 'Fin

C:\Users\58811.WMCDOMAIN\AppData\Local\Temp\ipykernel_34508\3854591763.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_schemas = pd.read_sql(schema_query, dev_conn)


In [79]:
# Create schema if it doesn't exist, then create table
create_schema_and_table = """
-- Create schema if it doesn't exist
IF NOT EXISTS (SELECT * FROM sys.schemas WHERE name = 'ClientAdmin')
BEGIN
    EXEC('CREATE SCHEMA ClientAdmin')
    PRINT 'Schema ClientAdmin created'
END
ELSE
BEGIN
    PRINT 'Schema ClientAdmin already exists'
END

-- Create table if it doesn't exist
IF NOT EXISTS (SELECT * FROM sys.objects WHERE object_id = OBJECT_ID(N'ClientAdmin.DownstreamTablesColumnsBASE') AND type in (N'U'))
BEGIN
    CREATE TABLE ClientAdmin.DownstreamTablesColumnsBASE (
        DestinationBindingID INT NOT NULL,
        ParsedDatabaseNM NVARCHAR(255) NULL,
        ParsedSchemaNM NVARCHAR(255) NULL,
        ParsedTableNM NVARCHAR(255) NULL,
        ParsedColumnNM NVARCHAR(255) NULL,
        ParsedColumnAliasNM NVARCHAR(255) NULL
    )
    PRINT 'Table ClientAdmin.DownstreamTablesColumnsBASE created'
END
ELSE
BEGIN
    PRINT 'Table ClientAdmin.DownstreamTablesColumnsBASE already exists'
END
"""

cursor = dev_conn.cursor()
cursor.execute(create_schema_and_table)
dev_conn.commit()
print("Schema and table check/creation completed")

Schema and table check/creation completed


In [80]:
# Clear existing data for these DestinationBindingIDs (optional)
binding_ids = df_output['DestinationBindingID'].unique()
binding_ids_str = ','.join([str(x) for x in binding_ids])

cursor = dev_conn.cursor()

# Check if table exists
check_query = "SELECT OBJECT_ID('ClientAdmin.DownstreamTablesColumnsBASE', 'U')"
cursor.execute(check_query)
table_exists = cursor.fetchone()[0] is not None

if table_exists:
    delete_query = f"""
    DELETE FROM ClientAdmin.DownstreamTablesColumnsBASE 
    WHERE DestinationBindingID IN ({binding_ids_str})
    """
    cursor.execute(delete_query)
    deleted_count = cursor.rowcount
    dev_conn.commit()
    print(f"Deleted {deleted_count} existing records")
else:
    print("Table does not exist yet - skipping delete")

Deleted 0 existing records


In [81]:
# Insert parsed data into destination table
insert_query = """
INSERT INTO ClientAdmin.DownstreamTablesColumnsBASE 
    (DestinationBindingID, ParsedDatabaseNM, ParsedSchemaNM, ParsedTableNM, ParsedColumnNM, ParsedColumnAliasNM)
VALUES (?, ?, ?, ?, ?, ?)
"""

# Replace NaN with None and convert to list of tuples
df_output_clean = df_output.fillna('')  # Use empty string instead of None

cursor = dev_conn.cursor()
inserted_count = 0

for idx, row in df_output_clean.iterrows():
    try:
        # Convert values explicitly to handle None/NaN
        values = (
            int(row['DestinationBindingID']),
            str(row['ParsedDatabaseNM']) if row['ParsedDatabaseNM'] else None,
            str(row['ParsedSchemaNM']) if row['ParsedSchemaNM'] else None,
            str(row['ParsedTableNM']) if row['ParsedTableNM'] else None,
            str(row['ParsedColumnNM']) if row['ParsedColumnNM'] else None,
            str(row['ParsedColumnAliasNM']) if row['ParsedColumnAliasNM'] else None
        )
        cursor.execute(insert_query, values)
        inserted_count += 1
    except Exception as e:
        print(f"Error on row {idx}: {e}")
        print(f"Values: {row.to_dict()}")
        break

dev_conn.commit()
print(f"Successfully inserted {inserted_count} records into ClientAdmin.DownstreamTablesColumnsBASE")

Successfully inserted 66 records into ClientAdmin.DownstreamTablesColumnsBASE


In [82]:
# Verify the insert
verify_query = f"""
SELECT TOP 10 * 
FROM ClientAdmin.DownstreamTablesColumnsBASE 
WHERE DestinationBindingID IN ({binding_ids_str})
"""

df_verify = pd.read_sql(verify_query, dev_conn)
print(f"Verification: Found {len(df_verify)} records in destination table")
df_verify

C:\Users\58811.WMCDOMAIN\AppData\Local\Temp\ipykernel_34508\1751774217.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_verify = pd.read_sql(verify_query, dev_conn)


Verification: Found 10 records in destination table


,DestinationBindingID,ParsedDatabaseNM,ParsedSchemaNM,ParsedTableNM,ParsedColumnNM,ParsedColumnAliasNM
0,2828,None,None,None,CAST,None
1,2828,None,None,None,PayerHierarchyCD,None
2,2828,None,None,None,PayerHierarchyCD,None
3,2828,None,None,None,PayerHierarchyCD,None
4,2828,None,None,None,ICDDiagnosisCD,None
5,2828,None,None,None,REPLACE,None
6,2763,None,None,None,PayerHierarchyCD,None
7,2763,None,None,None,PayerHierarchyCD,None
8,2764,None,None,None,PayerHierarchyCD,None
9,2805,None,None,None,PayerHierarchyCD,None


In [83]:
# Close connections
prod_conn.close()
dev_conn.close()
print("Connections closed")

Connections closed
